# Combined Notebook
This notebook contains all project scripts.

## crawler.py

In [ ]:
import logging
import requests
import re
import time
import os
from bs4 import BeautifulSoup
from collections import deque
from urllib.parse import urlparse, urljoin
from langdetect import detect
from urllib.robotparser import RobotFileParser
from datetime import datetime
import hashlib



def load_next_cycle_number():
    """
    Determine the next available crawling cycle number.

    Each cycle of crawling stores its pages under:
        data/cycle_XXX/<domain>/

    This function scans existing cycle folders and returns the next index.
    """
    base_path = "data"
    os.makedirs(base_path, exist_ok=True)

    cycles = [
        d for d in os.listdir(base_path)
        if d.startswith("cycle_") and d[6:].isdigit()
    ]

    if not cycles:
        return 1  # First cycle ever

    last = max(int(c[6:]) for c in cycles)
    return last + 1


# ----   Hash-based duplicate detection (same exact content) ----
SEEN_HASHES = set()

def compute_hash(text):
    """
    Compute MD5 hash of the page content.
    Used to skip saving duplicate pages across cycles.
    """
    return hashlib.md5(text.encode('utf-8')).hexdigest()

#  Logging setup: save all crawl events in crawler.log
logging.basicConfig(
    filename="crawler.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def is_greek(text: str) -> bool:
    """
    Simple heuristic Greek-language detector.
    Returns True if Greek characters are found.
    """

    sample = text[:3000]  # Check only first part of text
    greek_chars = sum(1 for c in sample if 'ά' <= c <= 'ώ' or 'Α' <= c <= 'Ω')
    total_chars = len(sample)

    # Rule 1: Enough Greek letters
    if greek_chars >= 20:
        return True

    # Rule 2: Proportion threshold
    if total_chars > 0 and (greek_chars / total_chars) >= 0.10:
        return True

    return False



#  HTML Text Cleaning Function
def clean_text(text: str) -> str:
    """
    Remove HTML noise and keep only useful Greek/English words.
    """
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    # Keep only Greek, Latin, digits and basic punctuation
    text = re.sub(r"[^A-Za-z\u0370-\u03FF0-9.,?!()'\" \n]", " ", text)

    # Remove isolated date/time-like numbers (π.χ. 14:32, 18/03/2025)
    text = re.sub(r"\b\d{1,2}:\d{1,2}\b", " ", text)
    text = re.sub(r"\b\d{1,2}/\d{1,2}/\d{4}\b", " ", text)

    # Remove repeated punctuation
    text = re.sub(r"[.,?!]{2,}", " ", text)

    return text.strip()


# Save cleaned text to disk inside the correct cycle/domain
def save_page_text(cycle_number, domain: str, url: str, text: str, date_string: str, counter: int) -> None:
    """
    Save a single cleaned page to:
        data/cycle_XXX/<domain>/page_N.txt

    Each file also stores URL and date metadata for traceability.
    """
    folder = os.path.join("data",f"cycle_{cycle_number:03d}", domain)
    os.makedirs(folder, exist_ok=True)  # Create folder per domain

    filename = os.path.join(folder, f"page_{counter}.txt")

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"URL={url}\n")            # Store URL traceability
            f.write(f"DATE={date_string}\n")   # Store date of crawling
            f.write(text)                      # Store cleaned text
        print(f"Saved: {filename}")
    except Exception as e:
        logging.error(f"Error saving page {url}: {e}")


# ---  robots.txt Compliance Helper ---
_robots_cache = {}


def is_allowed_by_robots(url: str, user_agent: str = "*") -> bool:
    """
    Check if crawling the given URL is allowed by robots.txt
    """
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

    # Cached download to avoid repeated robots.txt fetches
    if robots_url not in _robots_cache:
        rp = RobotFileParser()
        rp.set_url(robots_url)
        try:
            rp.read()
        except Exception:
            # Αν αποτύχει, default -> επιτρέπουμε crawl
            _robots_cache[robots_url] = None
            return True
        _robots_cache[robots_url] = rp

    rp = _robots_cache[robots_url]
    if rp is None:
        return True

    try:
        return rp.can_fetch(user_agent, url)
    except Exception:
        return True


# --- MAIN BFS CRAWLER ---
def crawl_bfs(cycle_number: int, starting_url: str, config: dict):
    """
    Perform BFS-based web crawling starting from a root URL.

    - Respects robots.txt
    - Extracts internal links only (same domain)
    - Applies depth limitation
    - Saves cleaned content
    - Logs term counts per page
    """
    visited = set()          # URLs that have been visited
    queued = set()           # URLs that are in the queue
    queue = deque()

    parsed_root = urlparse(starting_url)
    root_domain = parsed_root.netloc
    root_base = f"{parsed_root.scheme}://{parsed_root.netloc}"

    max_depth = config.get("max_depth", 2)
    delay = config.get("delay", 0.2)
    max_pages = config.get("max_pages", 50)

    pages_saved = 0  # how many pages have been saved so far

     # Initialize BFS queue
    queue.append((starting_url, 0))
    queued.add(starting_url)

    print(f"\n=== Starting crawl for domain: {root_domain} ===")

    while queue and pages_saved < max_pages:
        current_url, depth = queue.popleft()
        queued.discard(current_url)

        # Skip already visited or too deep
        if current_url in visited or depth > max_depth:
            continue

        visited.add(current_url)

        # robots.txt check
        if not is_allowed_by_robots(current_url, user_agent="GeorgiosCrawler/1.0"):
            print(f"Blocked by robots.txt -> {current_url}")
            logging.info(f"Blocked by robots.txt -> {current_url}")
            continue

        # Fetch page content(Attempt page download)
        try:
            response = requests.get(
                current_url,
                headers={"User-Agent": "GeorgiosCrawler/1.0"},
                timeout=10
            )
            response.raise_for_status()
            html = response.text
            soup = BeautifulSoup(html, "lxml")
        except Exception as e:
            logging.error(f"Failed to crawl {current_url}: {e}")
            continue

        # --- EXTRACT AND CLEAN TEXT ---

        #  Extract visible content from article-like containers
        article_body = soup.find("div", class_=re.compile(r"(article-body|entry-content|post-content)"))
        if not article_body:
            article_body = soup.find("article")
        if not article_body:
            article_body = soup.body

        if article_body:
            content_tags = article_body.find_all(["p", "h1", "h2", "h3"])
        else:
            content_tags = soup.find_all(["p", "h1", "h2", "h3"])

        texts = [tag.get_text(separator=" ", strip=True) for tag in content_tags]
        page_text = "\n".join(texts)
        page_text = clean_text(page_text)

        if not page_text.strip():
            # Αν δεν υπάρχει ουσιαστικό κείμενο, δεν σώζουμε
            logging.info(f"No text extracted for {current_url}")
            # συνεχίζουμε να μαζεύουμε links όμως
        else:
            # Date extraction (πολύ απλό – για εργασία αρκεί)
            date_string = datetime.now().strftime("%Y-%m-%d")
            
             # Greek-language filter
            if not is_greek(page_text):
                logging.info(f"[SKIP] Low-Greek-content page: {current_url}")
                continue

             # Extra langdetect confirmation
            try:
                lang = detect(page_text[:1000])
                if lang != "el":
                    logging.info(f"Skipping non-Greek page: {current_url} (detected lang={lang})")
                else:
                    # Avoid saving duplicate content using hash
                    content_hash = compute_hash(page_text)
                    if content_hash in SEEN_HASHES:
                        logging.info(f"[SKIP] Duplicate content detected at {current_url}")
                        continue
                        SEEN_HASHES.add(content_hash)

                    # Save Page
                    domain_name = root_domain.replace("www.", "")
                    pages_saved += 1

                    # Compute a simple "direct term metric" (#words)
                    direct_terms = len(page_text.split()) 
                    with open("crawler_event_log.csv", "a", encoding="utf-8") as f:
                        f.write(f"{datetime.now()},{direct_terms}\n")
                    print(f"Crawling {pages_saved}/{max_pages} (Depth {depth}) -> {current_url}")
                    save_page_text(cycle_number, domain_name, current_url, page_text, date_string, pages_saved)
            except Exception as e:
                logging.debug(f"Language detection failed for {current_url}: {e}")


            

        # --- EXTRACT & FILTER LINKS ---
        #Extract internal links and push them into the BFS queue
        for link in soup.find_all("a", href=True):
            href = link["href"]

            # Skip empty / javascript / anchors
            if not href or href.startswith("#") or href.lower().startswith("javascript:"):
                continue

            # Build absolute URL relative to the CURRENT PAGE
            new_url = urljoin(current_url, href)
            parsed_new = urlparse(new_url)

            # Keep only http/https
            if parsed_new.scheme not in ("http", "https"):
                continue

            # Keep only internal links of the same domain
            if parsed_new.netloc != root_domain:
                continue

            # Normalise (drop fragment)
            new_url = parsed_new._replace(fragment="").geturl()

            # Avoid duplicates: already visited OR already in queue
            if new_url in visited or new_url in queued:
                continue

            # Heuristic filters to skip non-article links
            path = parsed_new.path or "/"
            # we skip:
            #  - /page/x
            #  - paths with at least 2  slashes
            #  - paths that contains digits

            is_pagination = re.search(r"/page/\d+/?$", path) is not None
            has_digits = re.search(r"\d", path) is not None
            depth_path = len([p for p in path.split("/") if p])

            if not (is_pagination or has_digits or depth_path >= 2):
                
                continue

            # robots.txt check for the new link
            if not is_allowed_by_robots(new_url, user_agent="GeorgiosCrawler/1.0"):
                continue

            queue.append((new_url, depth + 1))
            queued.add(new_url)

        # Respect polite crawling delay
        time.sleep(delay)

    print(f"Finished crawling {root_domain}. Total visited: {len(visited)}, pages saved: {pages_saved}")
    return visited

#  MAIN EXECUTION LOOP — infinite cycles

if __name__ == "__main__":

    # Load URLs from seed.txt
    urls = []

    try:
        with open("seed.txt", "r", encoding="utf-8") as f:
            urls = [
                line.strip()
                for line in f.readlines()
                if line.strip() and not line.strip().startswith("#")
            ]
        print(f"URLs loaded from seed.txt:\n{urls}\n")
    except FileNotFoundError:
        print("Error: seed.txt not found. Please create the file.")
        exit(1)

    # Crawler configuration
    config = {
        "max_depth": 2,   # πόσα link-levels βαθιά
        "delay": 0.2,     # delay ανά request (seconds)
        "max_pages": 50   # πόσες σελίδες να ΣΩΣΕΙ ανά domain
    }
    
    #  Infinite crawling loop: each cycle generates new data/cycle_XXX/
    while True:
        cycle_number = load_next_cycle_number()
        print(f"\n=== Starting crawling cycle {cycle_number:03d} ===\n")

        for start_url in urls:
            crawl_bfs(cycle_number, start_url, config)

        print(f"\n=== Cycle {cycle_number:03d} finished. Sleeping for 1 hour... ===\n")
        time.sleep(3600)


## analysis.py

In [ ]:
# analysis.py
import os
import re
import csv
from datetime import datetime
from collections import defaultdict
from terms import TERMS   # Dictionary containing all categories and their associated keyword lists


# ============================================================
# 1. Preprocessing helpers
# ============================================================

def remove_boilerplate(text: str) -> str:
    """
    Removes common boilerplate text that frequently appears in news articles,
    such as cookie policy, copyright disclaimers, newsletter messages, etc.
    This helps reduce noise and prevents irrelevant words from affecting
    the keyword frequency analysis.
    """
    phrases = [
        "όροι χρήσης",
        "όροι και προϋποθέσεις",
        "πολιτική απορρήτου",
        "πολιτική cookies",
        "χρήση cookies",
        "all rights reserved",
        "copyright",
        "newsletter",
        "εγγραφείτε στο newsletter",
        "μήνυμα αποποίησης",
        "disclaimer",
        "διαβάστε περισσότερα εδώ",
    ]
    for ph in phrases:
        text = text.replace(ph, " ")
    return text


def preprocess_text(text: str) -> str:
    """
    Full text preprocessing pipeline applied to every crawled page:
    - lowercasing
    - removing HTML tags and encoded entities (&nbsp;, &gt;, etc.)
    - removing boilerplate / disclaimers
    - keeping only Greek/Latin letters, digits, and spaces
    - compressing multiple spaces into one

    The result is a clean, normalized text suitable for keyword matching.
    """
    
    text = text.lower()

     # remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text) 

    # remove HTML entities
    text = re.sub(r"&[a-z]+;", " ", text)

    # boilerplate
    text = remove_boilerplate(text)

    # keep only allowed characters
    text = re.sub(
        r"[^0-9a-zάέήίόύώϊΐϋΰα-ω ]+",
        " ",
        text,
        flags=re.IGNORECASE
    )

     # compress extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_term_pattern(term: str) -> re.Pattern:
    """
    Creates a compiled regex pattern for a given term.
    This includes:
    - escaping special characters
    - replacing spaces with \s+ to match any spacing variation
    - allowing dashes instead of spaces
    - applying word boundaries to avoid partial matches

    Regex compilation makes term matching significantly faster.
    """
    base = term.lower().strip()
    base = re.escape(base)
    base = base.replace(r"\ ", r"\s+")
    base = base.replace(r"\-", r"[-\s]?")
    pattern = r"\b" + base + r"\b"
    return re.compile(pattern, flags=re.IGNORECASE)


# ============================================================
#  2. Load all pages from /data/
# ============================================================

def load_pages():
    """
    Loads every crawled page stored under /data/.
    Each file contains:
        URL=<url>
        DATE=<crawl_date>
        <page content>

    This function extracts:
    - domain
    - file name
    - URL
    - crawl date
    - preprocessed content

    Returns:
        A list of dictionaries, each representing one crawled page.
    """

    pages = []

    # Walk through all folders inside /data/ (including cycle folders)
    for root, dirs, files in os.walk("data"):
        for file in files:
            if not file.endswith(".txt"):
                continue

            filepath = os.path.join(root, file)

            try:
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    lines = f.readlines()
            except Exception as e:
                print(f"ΣΦΑΛΜΑ ανάγνωσης αρχείου {filepath}: {e}")
                continue

            if not lines:
                continue

            # Extract URL from first line if present
            first = lines[0].strip()

            if first.startswith("URL="):
                url = first.replace("URL=", "").strip()
                body_start_idx = 1

            else:
                url = "Unknown URL"
                body_start_idx = 0

            # Extract DATE from second line if present
            date_string = None
            if body_start_idx < len(lines) and lines[body_start_idx].startswith("DATE="):
                date_string = (
                    lines[body_start_idx]
                    .strip()
                    .replace("DATE=", "")
                    .strip()
                )
                body_start_idx += 1

            # If date not found inside the file, use file modification timestamp
            if not date_string:
                ts = os.path.getmtime(filepath)
                date_string = datetime.fromtimestamp(ts).strftime("%Y-%m-%d")

            # Remaining lines are page content
            raw_content = " ".join(l.strip() for l in lines[body_start_idx:])
            content = preprocess_text(raw_content)
            
            # Use folder name as domain
            domain = os.path.basename(root)

            pages.append(
                {
                    "domain": domain,
                    "file": file,
                    "url": url,
                    "date": date_string,
                    "content": content,
                }
            )

    return pages


# ============================================================
# 3. Main analysis: count terms, generate time series and totals
# ============================================================

def analyze_all_pages():
    """
    Main analysis pipeline:
    - Load all pages from /data/
    - Precompile regex patterns for all terms
    - Count occurrences per category and per term
    - Build a daily time series (DATE → category counts)
    - Save results into two CSV files
        * analysis_timeseries.csv
        * analysis_totals.csv
    """
    all_pages = load_pages()

    print(f"\nΞεκινά η ανάλυση {len(all_pages)} σελίδων...")
    print("-" * 40)

     # Precompile regex patterns for speed

    term_patterns = {}

    for category, terms_list in TERMS.items():
        for term in terms_list:
            if term not in term_patterns:
                term_patterns[term] = build_term_pattern(term)

    print("\nOK: Έτοιμα τα compiled regex patterns.\n")

    # Daily results: counts per day per category
    daily_results = defaultdict(lambda: {cat: 0 for cat in TERMS.keys()})

    # Total counts per category
    total_category_counts = {cat: 0 for cat in TERMS.keys()}

    # Total counts per individual keyword
    all_terms = [t for lst in TERMS.values() for t in lst]
    total_term_counts = {t: 0 for t in all_terms}

    # ====================================================
    # Iterate through every crawled page
    # ====================================================
    for page in all_pages:
        text = page["content"]
        page_date = page["date"]

        # Count keyword occurrences per category
        for category, terms_list in TERMS.items():
            current_category_count = 0

            for term in terms_list:
                pattern = term_patterns[term]
                matches = pattern.findall(text)
                count = len(matches)

                if count > 0:
                    current_category_count += count
                    total_term_counts[term] += count

            # Update daily time series and totals
            daily_results[page_date][category] += current_category_count
            total_category_counts[category] += current_category_count

    # ====================================================
    # Print totals to console for inspection
    # ====================================================
    print("\n=== ΣΥΝΟΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ ===\n")

    print("Ανά κατηγορία:")
    for cat, v in total_category_counts.items():
        print(f" - {cat}: {v} εμφανίσεις")

    print("\nΑνά όρο:")
    for term, v in total_term_counts.items():
        if v > 0:
            print(f" - '{term}': {v} εμφανίσεις")

    # ====================================================
    #  Save time series CSV (Date, Category1, Category2, ...)
    # ====================================================
    relevant_dates = list(daily_results.keys())
    sorted_dates = sorted(relevant_dates)
    category_names = list(TERMS.keys())

    timeseries_csv = "analysis_timeseries.csv"
    with open(timeseries_csv, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["Date"] + category_names
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for date_key in sorted_dates:
            row = {"Date": date_key}
            row.update(daily_results[date_key])
            writer.writerow(row)

    print(f"\n[CSV] Τα αποτελέσματα χρονοσειράς αποθηκεύτηκαν στο: {timeseries_csv}")

    # ====================================================
    # # Save totals CSV (more flexible format for plotting)
    # ====================================================
    totals_csv = "analysis_totals.csv"
    with open(totals_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Type", "Name", "Count"])

        # Category totals
        for cat, v in total_category_counts.items():
            writer.writerow(["Category", cat, v])

         # Term totals
        for term, v in total_term_counts.items():
            writer.writerow(["Term", term, v])

    print(f"[CSV] Τα συνολικά αποτελέσματα αποθηκεύτηκαν στο: {totals_csv}\n")

    print("\nΑνάλυση ολοκληρώθηκε!\n")

    # Αν θέλεις να το χρησιμοποιήσεις από άλλα scripts (π.χ. charts.py),
    # μπορείς να το κάνεις import και να πάρεις αυτά τα αποτελέσματα:
    return daily_results, total_category_counts, total_term_counts


# ============================================================
# 6. Script entrypoint
# ============================================================
if __name__ == "__main__":
    analyze_all_pages()


## charts.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------
# 1. BAR CHART: Total occurrences per category
# ---------------------------------------------------------

# Load the summary CSV generated by analysis.py.
# The file contains rows of type:
#   - Category:  aggregated counts for each category
#   - Term:      aggregated counts for each individual keyword
#
# Here we extract only the category-level rows in order to
# visualize how frequently each category appears across all
# crawled documents.

totals = pd.read_csv("analysis_totals.csv")

category_data = totals[totals["Type"] == "Category"].copy()

# Create a vertical bar chart showing total category counts.
plt.figure(figsize=(12,6))
plt.bar(category_data["Name"], category_data["Count"], width = 0.3, color = "darkblue")

plt.title("Συνολικές Εμφανίσεις ανά Κατηγορία", fontsize=16)
plt.ylabel("Εμφανίσεις", fontsize=10)

# Rotate category labels so long names stay readable
plt.xticks(rotation=45, ha="right", fontsize = 5)

plt.tight_layout()
plt.grid(axis = 'y', alpha = 0.6)

plt.show()

# ---------------------------------------------------------
# 2. HORIZONTAL BAR CHART: Keyword appearances (only >0)
# ---------------------------------------------------------

# Extract only term-level rows (individual keyword counts).
term_data = totals[totals["Type"] == "Term"].copy()

# Keep only terms that actually appeared in the data.
# (Helps remove noise from unused keywords.)
term_data = term_data[term_data["Count"] > 0]     

# Sort in ascending order so the horizontal bar chart reads nicely
term_data = term_data.sort_values("Count", ascending=True)   

# Create a horizontal bar chart to display keyword frequencies.
plt.figure(figsize=(12,10))
plt.barh(term_data["Name"], term_data["Count"])

plt.title("Συνολικές Εμφανίσεις ανά Όρο (Keyword)", fontsize=16)
plt.xlabel("Εμφανίσεις", fontsize=14)

plt.tight_layout()
plt.grid(axis = 'x')

plt.show()






## generate_plots.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the log file created by the crawler.
# Each row contains:
#   timestamp → when a page was crawled
#   direct_terms → how many matching terms were found in that page
df = pd.read_csv(
    "crawler_event_log.csv",
    header=None,
    names=["timestamp", "direct_terms"]
)

# Convert the timestamp column to actual datetime objects
# so that Matplotlib can plot them properly on the x-axis.
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort chronologically in case rows were written slightly out of order.
df = df.sort_values("timestamp")

# Add a new column: cumulative sum of terms across time.
# This helps us observe long-term growth and whether the crawler
# continues discovering new content.
df["cumulative_terms"] = df["direct_terms"].cumsum()


# Plot 1 — Cumulative number of terms over time
# This shows how fast the crawler collects term occurrences
# as it processes more pages across cycles.
plt.figure(figsize=(12, 5))
plt.plot(df["timestamp"], df["cumulative_terms"])
plt.title("Cumulative Terms Found Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Cumulative Terms Found")
plt.grid(True)
plt.tight_layout()
plt.show()


# Plot 2 — Terms detected per individual crawled page
# This visualizes the variability: some pages have many terms,
# some very few. It reveals spikes or unusual patterns that could
# indicate potentially suspicious news articles.
plt.figure(figsize=(12, 5))
plt.plot(df["timestamp"], df["direct_terms"])
plt.title("Direct Terms Found Per Page (Over Time)")
plt.xlabel("Timestamp")
plt.ylabel("Direct Terms Found")
plt.grid(True)
plt.tight_layout()
plt.show()


## terms.py

In [ ]:
TERMS = {

    "Παροχή Επενδυτικών Υπηρεσιών και Υπηρεσιών σε κρυπτοστοιχεία": [
        
        "επένδυση", "επενδυτική", "κρυπτο", "crypto",
        "κεφάλαιο", "διαχείριση", "χαρτοφυλάκιο",
        "απόδοση", "αποδόσεις", "συμβουλή",
        "ρίσκο", "μετοχη", "ομόλογα", "fund", "ETF", "broker",
        
        "investment", "portfolio", "asset", "market", "analysis",
        "financial", "equity", "securities", "derivatives",
    ],

    "Red flags και φράσεις πίεσης": [
        "μόνο.*σήμερα", "limited time", "προσφορά",
        "επενδύστε τώρα", "buy now", "άμεσα", "επείγον",
        "υψηλό κέρδος", "100% εγγυημένο", "χωρίς ρίσκο",
        "guaranteed return", "risk-free", "get rich",
        "γρήγορο κέρδος", "σίγουρο κέρδος",
    ],

    "Κρυπτο-υπηρεσίες": [
        "crypto", "token", "ICO", "exchange", "mining",
        "bitcoin", "ethereum", "altcoin", "blockchain",
        "NFT", "wallet", "metaverse",
        
        "ψηφιακό νόμισμα", "ψηφιακο", "κρυπτονομισμα", 
    ],

    "Έμμεσοι Δείκτες": [
        "πληρωμές μόνο με κρυπτονομίσματα",
        "απαγορεύονται οι τραπεζικές συναλλαγές",
        "αποφυγή τραπεζικού συστήματος",
        "ανώνυμοι διαχειριστές ή συντονιστές",
        "Μη αναφορά σε στοιχεία επικοινωνίας ή επίσημη έδρα",
        "Ασυνεπείς όροι σε διαφορετικές σελίδες (FAQ vs Terms) - δείχνει προσωρινότητα / copy-paste."
    ]
}



## Expanded List (New Terms Added)

### Category 1 — Suspicious Price Movements
`"αιφνίδια άνοδος", "αιφνίδια πτώση", "ξαφνική μεταβολή", "έντονη διακύμανση", "μεταβλητότητα αγοράς", "απότομη διόρθωση", "έντονο ανοδικό ράλι", "ξεπούλημα μετοχών", "απότομη ξεπούλημα", "σπασμωδικές κινήσεις", "υπέρμετρη άνοδος", "υπέρμετρη πτώση", "αδικαιολόγητη μεταβολή", "μη φυσιολογική κίνηση", "ασυνήθιστη μεταβλητότητα"`

### Category 2 — Insider Trading / Information Leaks
`"διαρροή πληροφοριών", "εμπιστευτικές πληροφορίες", "προνομιακή ενημέρωση", "εσωτερική πληροφόρηση", "ύποπτη συναλλαγή πριν την ανακοίνωση", "συναλλαγές πριν τη δημοσίευση", "συναλλαγές με προνομιακά δεδομένα", "εσωτερική εκμετάλλευση", "ύποπτη συμπεριφορά μετόχου", "ανεξήγητη δραστηριότητα πριν τα νέα"`

### Category 3 — Market Manipulation Tactics
`"τεχνητός όγκος", "τεχνητή ζήτηση", "τεχνητή προσφορά", "χειραγώγηση τιμής", "pump and dump", "wash trading", "spoofing", "layering", "market cornering", "trade-based manipulation", "quote stuffing", "πλασματικές εντολές", "πλασματικός όγκος"`

### Category 4 — Regulatory / Legal Alerts
`"παρέμβαση της επιτροπής κεφαλαιαγοράς", "ρυθμιστική προειδοποίηση", "έρευνα αγοράς", "έλεγχος συναλλαγών", "ανακοίνωση εποπτικής αρχής", "παράβαση κανόνων", "παρατυπία αγοράς", "προειδοποίηση εποπτείας", "καταγγελία χειραγώγησης", "πειθαρχική διαδικασία", "αναστολή διαπραγμάτευσης"`




### Prompt Used (ChatGPT gpt-5.1)

“Expand each list of market-manipulation related keywords by adding more synonyms, semantically related expressions, typical phrases used in financial journalism, and morphological variations.

Keep all terms strictly relevant to stock-market manipulation, investor sentiment, suspicious trading activity, sudden movements, regulatory alerts, and market abuse.

Do NOT include terms that are too generic, unrelated to finance, or that would produce false positives.”"


---

## Methodology for Expanding the Keyword Lists

1. **Initial Seed Terms**  
   Started from the confidential list provided in the assignment.

2. **Semantic Expansion**  
   Added synonyms, related financial terms, morphological variants, and phrase variations.

3. **Contextual Relevance Check**  
   Kept only terms relevant to Greek financial news and the manipulation‑related context.

4. **Noise Filtering**  
   Removed overly generic terms to avoid false positives.

5. **Use of ChatGPT (gpt‑5.1)**  
   ChatGPT was used strictly as a *support tool* to propose linguistically similar terms.  
   The final selection was manually reviewed to ensure contextual correctness.



## Displaying `analysis_timeseries.csv`

In [ ]:
import pandas as pd
df = pd.read_csv("analysis_timeseries.csv")
df.head()

## Displaying `analysis_totals.csv`

In [ ]:
import pandas as pd
df = pd.read_csv("analysis_totals.csv")
df.head()

## Displaying `crawler_event_log.csv`

In [ ]:
import pandas as pd
df = pd.read_csv("crawler_event_log.csv")
df.head()